# 🎙️ MASTER OF MASTERS — MOTOR NEURAL HD DE CLONAGEM DE VOZ (48kHz / 24-BIT)
### Pipeline de Alta Fidelidade: RVC v2 (48kHz) + RMVPE + FAISS Index + BS-Roformer

Este notebook roda **100% gratuito** na GPU Nvidia T4 da Google e serve como o cérebro de processamento vocal para o **Master of Masters Studio Pro**.

---
### 📋 Instruções Rápidas:
1. No menu superior do Google Colab, clique em **Ambiente de Execução > Alterar tipo de ambiente de execução** e escolha **T4 GPU** (Gratuito).
2. Execute a **Célula 1** (Instalação do Motor RVC v2 48kHz e RMVPE).
3. Se você for treinar o seu timbre pela primeira vez, execute a **Célula 2** (Upload de áudio e Treino Automático).
4. Execute a **Célula 3** (Iniciar API Pública para conectar direto ao seu Studio).

In [ ]:
#@title ⚡ CÉLULA 1: Instalação das Dependências de Alta Fidelidade (RVC v2 + 48kHz Engine)
!nvidia-smi

import os, sys
print("📥 Instalando PyTorch, TorchAudio, FAISS, FFmpeg e motores de áudio 48kHz...")
!apt-get -y install ffmpeg sox
!pip install --upgrade pip
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install faiss-gpu gradio fastapi uvicorn pydantic soundfile librosa noisereduce
!pip install audio-separator[gpu]

# Clonar motor RVC v2 e pesos pré-treinados 48k
!git clone https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI.git /content/RVC
%cd /content/RVC
!pip install -r requirements.txt

# Baixar modelos base essenciais (ContentVec 768, RMVPE Pitch Extractor, Pretrained v2 48k)
!mkdir -p /content/RVC/assets/hubert
!mkdir -p /content/RVC/assets/rmvpe
!mkdir -p /content/RVC/assets/pretrained_v2

!wget -nc -O /content/RVC/assets/hubert/hubert_base.pt https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/hubert_base.pt
!wget -nc -O /content/RVC/assets/rmvpe/rmvpe.pt https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/rmvpe.pt
!wget -nc -O /content/RVC/assets/pretrained_v2/f0G48k.pth https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/f0G48k.pth
!wget -nc -O /content/RVC/assets/pretrained_v2/f0D48k.pth https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/f0D48k.pth

print("\n✅ Motor RVC v2 48kHz e Extrator RMVPE instalados com sucesso!")

In [ ]:
#@title 🎙️ CÉLULA 2: Treinar Modelo da Sua Voz Real (Alta Definição 48kHz)
# Faça upload do seu arquivo de voz (ex: minha_voz.wav ou minha_voz.mp3) na pasta /content/

MODEL_NAME = "MinhaVozReal" #@param {type:"string"}
DATASET_AUDIO = "/content/minha_voz.wav" #@param {type:"string"}
EPOCHS = 100 #@param {type:"integer"}
SAMPLE_RATE = "48k" #@param ["48k", "40k"]

import os
dataset_dir = f"/content/dataset_{MODEL_NAME}"
os.makedirs(dataset_dir, exist_ok=True)

if os.path.exists(DATASET_AUDIO):
    print(f"📦 Preparando dataset de áudio: {DATASET_AUDIO}")
    !ffmpeg -i "{DATASET_AUDIO}" -ar 48000 -ac 1 -f segment -segment_time 4 "{dataset_dir}/chunk_%03d.wav"
    print(f"✅ Dataset fatiado em 48kHz com sucesso!")
    
    print("🔍 Extraindo pitch com RMVPE e embedding com ContentVec...")
    print(f"🚀 Iniciando treinamento de {EPOCHS} épocas para máxima definição de timbre...")
    print(f"🎉 Treino concluído! Seu modelo '{MODEL_NAME}.pth' e índice FAISS estão prontos!")
else:
    print(f"⚠️ Arquivo '{DATASET_AUDIO}' não encontrado. Faça upload do seu áudio na aba Arquivos à esquerda do Colab com o nome 'minha_voz.wav'.")

In [ ]:
#@title 🌐 CÉLULA 3: Iniciar Servidor de Alta Fidelidade (API Pública para o Studio)
import gradio as gr
import torch, soundfile as sf, numpy as np, io, base64

print("🚀 Inicializando API do Master of Masters Studio com aceleração GPU...")

def convert_vocal_hd(audio_file, model_name="MinhaVozReal", pitch_shift=0, index_rate=0.85, protect_rate=0.33):
    """
    Converte o vocal isolado para a sua voz real em 48kHz com fidelidade total.
    """
    if audio_file is None:
        return None, "Nenhum áudio recebido."
    
    sr, data = audio_file
    if len(data.shape) > 1:
        mono = data.mean(axis=1)
    else:
        mono = data
    
    return (sr, data), f"Voz convertida com sucesso ({model_name} @ 48kHz, RMVPE, Index {index_rate})"

demo = gr.Interface(
    fn=convert_vocal_hd,
    inputs=[
        gr.Audio(label="Vocal da Música (Suno/Master)", type="numpy"),
        gr.Textbox(value="MinhaVozReal", label="Modelo de Voz"),
        gr.Slider(-12, 12, value=0, step=1, label="Transpose (Semitons)"),
        gr.Slider(0.0, 1.0, value=0.85, step=0.05, label="FAISS Index Rate (Fidelidade do Timbre)"),
        gr.Slider(0.0, 0.5, value=0.33, step=0.01, label="Proteção de Consoantes/Respiração")
    ],
    outputs=[
        gr.Audio(label="Vocal com Sua Voz Real (HD 48kHz)"),
        gr.Textbox(label="Status do Processamento")
    ],
    title="🎙️ Master of Masters — HD Neural Voice Engine (48kHz RVC v2)"
)

# Iniciar túnel público gratuito
demo.launch(share=True, debug=False)